![Databricks Academy](./Includes/images/common/db-academy.png)

# 5 - Implantando um Pipeline em Produção

Nesta demonstração, começaremos adicionando uma fonte de dados adicional ao nosso pipeline e realizando um join com nossas tabelas de streaming. Em seguida, focaremos na produção do pipeline, adicionando comentários e propriedades de tabela aos objetos criados, agendando o pipeline e criando um log de eventos para monitorar o pipeline.

### Objetivos de Aprendizagem

Ao final desta lição, você será capaz de:
- Aplicar a sintaxe de comentários apropriada e propriedades de tabela aos objetos do pipeline para melhorar a legibilidade.
- Demonstrar como realizar um join entre duas tabelas de streaming usando uma visão materializada para otimizar o processamento de dados.
- Executar o agendamento de um pipeline usando modos de trigger ou contínuo para garantir o processamento oportuno.
- Explorar o log de eventos para monitorar um pipeline de produção Lakeflow Spark Declarative.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**  
![Serverless Select](./Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook was **developed and tested using Serverless V5**. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the following cell to configure your working environment for this course.

    This cell will also reset your `/Volumes/labuser/sdp_1_bronze/source` volume with the JSON files to the starting point, with one JSON file in each directory.

In [0]:
%run ./Includes/Classroom-Setup-REQUIRED

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.8/832.8 kB 30.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e72d61ec-80b8-4b0a-b1ea-7f3527b072cc
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e72d61ec-80b8-4b0a-b1ea-7f3527b072cc
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser15140516_1778971530'.



  STEP 1: Verifying catalog exists: labuser15140516_1778971530
  Catalog 'labuser15140516_1778971530' exists.

  STEP 2: Setting up 3 schema(s) in catalog: labuser15140516_1778971530
  [1/3] Checking: `labuser15140516_1778971530`.`sdp_1_bronze`... ALREADY EXISTS
  [2/3] Checking: `labuser15140516_1778971530`.`sdp_2_silver`... ALREADY EXISTS
  [3/3] Checking: `labuser15140516_1778971530`.`sdp_3_gold`... ALREADY EXISTS

  COMPLETE: 0 schema(s) created, 3 already existed.



DataFrame[]


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating volume folder path...
  Found: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers

  STEP 2: Scanning for files...
  Found 1 file(s) to delete.

  STEP 3: Deleting files from: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers
  [1/1] Deleting: 00.json... DELETED

  COMPLETE: Deleted 1 file(s) from /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/customers


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/la

Information,Value
Your Catalog:,labuser15140516_1778971530
Bronze Schema:,sdp_1_bronze
Silver Schema:,sdp_2_silver
Gold Schema:,sdp_3_gold
Source Volume:,/Volumes/labuser15140516_1778971530/sdp_1_bronze/source


Compute,Status,Details
Serverless,✓ Match,Version 5


## B. Explore the Orders and Status JSON Files

1. Explore the raw data located in the `/Volumes/labuser/sdp_1_bronze/source/orders/` volume.

   This is the data we have been working with throughout the course demonstrations.

   Run the cell below to view the results. Notice that the orders JSON file(s) contains information about when each order was placed.

In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/orders/',
  format => 'JSON'
)
LIMIT 10;

customer_id,notifications,order_id,order_timestamp,_rescued_data
23094,Y,75123,1640392092,null
23457,N,75124,1640392500,null
23564,Y,75125,1640394862,null
23392,N,75126,1640396067,null
23101,Y,75127,1640399066,null
23466,N,75128,1640404853,null
23834,Y,75129,1640407272,null
23852,Y,75130,1640419989,null
23483,Y,75131,1640422131,null
23821,N,75132,1640423697,null


2. Explore the **status** raw data located in the `/Volumes/labuser/sdp_1_bronze/source/status/` volume and filter for the specific **order_id** *75123*.

   Run the cell below to view the results. Notice that the status JSON file(s) contain **order_status** information for each order.

   **NOTE:** The **order_status** can include multiple rows per order and may be any of the following:

   - on the way
   - canceled
   - return canceled
   - reported shipping error
   - delivered
   - return processed
   - return picked up
   - placed
   - preparing
   - return requested


In [0]:
SELECT *
FROM read_files(
  source_volume_path || '/status/',
  format => 'JSON'
)
WHERE order_id = 75123;

order_id,order_status,status_timestamp,_rescued_data
75123,placed,1640392092,null
75123,preparing,1640733966,null
75123,on the way,1640549318,null
75123,delivered,1640604626,null


3. One of our objectives is to join the **orders** data with the order **status** data.

    The query below demonstrates what the result of the final join in the Spark Declarative Pipeline will look like after the data has been incrementally ingested and cleaned when we create the pipeline. Run the cell and review the output.

    Notice that after joining the tables, we can see each **order_id** along with its original **order_timestamp** and the **order_status** at specific points in time.

**NOTE:** The data used in this demo is artificially generated, so the **order_status_timestamps** may not reflect realistic timing.

In [0]:
WITH orders AS (
  SELECT *
  FROM read_files(
        source_volume_path || '/orders/',
        format => 'JSON'
  )
),
status AS (
  SELECT *
  FROM read_files(
        source_volume_path || '/status/',
        format => 'JSON'
  )
)
-- Join the views to get the order history with status
SELECT
  orders.order_id,
  timestamp(orders.order_timestamp) AS order_timestamp,
  status.order_status,
  timestamp(status.status_timestamp) AS order_status_timestamp
FROM orders
  INNER JOIN status
  ON orders.order_id = status.order_id
ORDER BY order_id, order_status_timestamp;

order_id,order_timestamp,order_status,order_status_timestamp
75123,2021-12-25T00:28:12.000Z,placed,2021-12-25T00:28:12.000Z
75123,2021-12-25T00:28:12.000Z,on the way,2021-12-26T20:08:38.000Z
75123,2021-12-25T00:28:12.000Z,delivered,2021-12-27T11:30:26.000Z
75123,2021-12-25T00:28:12.000Z,preparing,2021-12-28T23:26:06.000Z
75124,2021-12-25T00:35:00.000Z,placed,2021-12-25T00:35:00.000Z
75124,2021-12-25T00:35:00.000Z,preparing,2021-12-27T13:20:27.000Z
75124,2021-12-25T00:35:00.000Z,on the way,2021-12-27T14:11:48.000Z
75124,2021-12-25T00:35:00.000Z,delivered,2021-12-27T15:53:42.000Z
75125,2021-12-25T01:14:22.000Z,placed,2021-12-25T01:14:22.000Z
75125,2021-12-25T01:14:22.000Z,delivered,2021-12-26T20:37:20.000Z


## C. Putting a Pipeline in Production

This course includes a complete Lakeflow Spark Declarative Pipeline project that has already been created.

In this section, you'll explore the Spark Declarative Pipeline and modify its settings for production use.


1. The screenshot below shows what the final Spark Declarative Pipeline will look like when ingesting a single JSON file from the data sources:
![Final Demo 6 Pipeline](./Includes/images/deploying-a-pipeline-to-production/demo5_pipeline_image_run1.png)

    **Note:** Depending on the number of files you've ingested, the row count may vary.

2. Run the cell below to create your starter Spark Declarative Pipeline for this demonstration. The pipeline will set the following for you:
    - Your default catalog: **labuser**
    - Your configuration parameter: `source` = `/Volumes/labuser/sdp_1_bronze/source`

    **NOTE:** If the pipeline already exists, an error will be returned. In that case, you'll need to delete the existing pipeline and rerun this cell.

    To delete the pipeline:

    a. Select **Jobs & Pipelines** from the far-left navigation bar.

    b. Find the pipeline you want to delete.

    c. Click the three-dot menu ![ellipsis icon](./Includes/images/common/ellipsis_icon.png).

    d. Select **Delete**.

**NOTE:**  The `create_declarative_pipeline` function is a custom function built for this course to create the sample pipeline using the Databricks REST API. This avoids manually creating the pipeline and referencing the pipeline assets.

In [0]:
%python
create_declarative_pipeline(
    pipeline_name=f'5 - Deploying a Pipeline to Production Project - {my_catalog}',
    root_path_folder_name='5 - Deploying a Pipeline to Production Project',
    catalog_name=my_catalog,
    schema_name='default',
    source_folder_names=['orders', 'status'],
    configuration={'source': source_volume_path}
)


  STEP 1: Checking for existing pipeline...
  ✅ No existing pipeline named '5 - Deploying a Pipeline to Production Project - labuser15140516_1778971530' found.

  STEP 2: Building pipeline configuration...
  Pipeline Name:    5 - Deploying a Pipeline to Production Project - labuser15140516_1778971530
  Catalog:          labuser15140516_1778971530
  Schema:           default
  Root Path:        /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/5 - Deploying a Pipeline to Production Project
  Source Folders:    orders, status
  Serverless:       True
  Photon:           True
  Channel:          CURRENT
  Continuous:       False
  Development Mode: True
  Configuration:
    source = /Volumes/labuser15140516_1778971530/sdp_1_bronze/source

  STEP 3: Creating pipeline via API...
  ✅ Pipeline '5 - Deploying a Pipeline to Production Project - labuse

3. Complete the following steps to open the starter Spark Declarative Pipeline project for this demonstration:

   a. In the main navigation bar, right-click on **Jobs & Pipelines** and select **Open Link in New Tab**.

   b. In **Jobs & Pipelines** select your **5 - Deploying a Pipeline to Production Project - labuser** pipeline.
      - **REQUIRED:** At the top near your pipeline name, turn on **New pipeline monitoring**.

   c. In the **Pipeline details** pane on the far right select **Open in Editor** (field to the right of **Source code**) to open the pipeline in the **Lakeflow Pipeline Editor**.

   d. In the new tab, you should see the following folders:
      - **explorations**
      - **orders**
      - **status**
      - plus the extra **python_excluded** folder that contains the Python version.

## D. Explore the code in the `orders/orders_pipeline.sql` file

1. In your **Spark Declarative Pipeline** select the **orders** folder.

2. This file contains the same **orders_pipeline.sql** pipeline you've been working with throughout the course.

3. Quickly review the code again. Notice the following:
    - Each streaming table or materialized view now includes a `COMMENT` and `TBLPROPERTIES` section to document each object.
    - Data quality expectations were added from the previous demonstration.

## E. Explore the code in the `status/status_pipeline` notebook

### E1. Open the `status/status_pipeline.sql` file
1. In your Spark Declarative Pipeline editor open the **status/status_pipeline.sql** file.

2. This file processes new data and adds it to the pipeline for order **status**.

### E2. Criação da Tabela Bronze Status (`1_bronze_db.status_bronze_demo5`)

Use o código na seção `A. Bronze Table Creation` para entender como a tabela Bronze de streaming é criada e como a qualidade dos dados é aplicada.

1. Esta instrução cria a tabela de streaming **1_bronze_db.status_bronze_demo5** ao ingerir arquivos JSON brutos do caminho do volume do laboratório: `/Volumes/labuser_/sdp_1_bronze/source/status/`

2. A cláusula `COMMENT` adiciona metadados descritivos à tabela, facilitando o entendimento do propósito da tabela ao navegar no Unity Catalog.

3. A seção `TBLPROPERTIES` adiciona configurações de nível de tabela:
   - `"quality" = "bronze"` rotula esta tabela como parte da camada Bronze na arquitetura medallion.
   - `"pipelines.reset.allowed" = false` impede atualizações completas da tabela, ajudando a evitar truncamento acidental e perda de checkpoints durante resets do pipeline.

4. A variável `${source}` na cláusula `FROM STREAM` é usada para referenciar dinamicamente o local do volume específico usando o parâmetro de configuração SDP.

**NOTAS**:
   - **Propriedades de tabela do pipeline**: [AWS](https://docs.databricks.com/aws/en/ldp/properties#pipeline-table-properties) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/properties#pipeline-table-properties) |
   [GCP](https://docs.databricks.com/gcp/en/ldp/properties#pipeline-table-properties)
   
   - **Para marcação adequada, visite Aplicar tags a objetos do Unity Catalog**: [AWS](https://docs.databricks.com/aws/en/database-objects/tags) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/database-objects/tags) |
   [GCP](https://docs.databricks.com/gcp/en/database-objects/tags). 
      - Marcação está fora do escopo deste curso.

### E3. Criação da Tabela Silver Status (`2_silver_db.status_silver_demo5`)

Use o código na seção `B. Bronze -> Silver` para entender como a tabela de streaming Silver é criada e como a qualidade dos dados é aplicada.

1. Esta instrução cria a tabela de streaming **2_silver_db.status_silver_demo5** a partir da tabela Bronze de streaming **1_bronze_db.status_bronze_demo5**.

2. A cláusula `SELECT`:
   - Seleciona apenas as colunas necessárias para a camada Silver.
   - Faz o cast de `status_timestamp` para timestamp como `order_status_timestamp`.

3. As cláusulas `CONSTRAINT` definem expectativas de qualidade de dados:
   - `valid_timestamp` descarta linhas onde o timestamp não é válido.
   - `valid_order_status` alerta quando o status não está na lista permitida.

4. As seções `COMMENT` e `TBLPROPERTIES` documentam a tabela e a rotulam como Silver na arquitetura medallion.

### E4. Visão Materializada para Juntar Duas Tabelas de Streaming (`3_gold_db.full_order_info_gold_demo5`)

> Uma forma de juntar duas tabelas de streaming em Spark Declarative Pipelines é criando uma visão materializada que realiza o join.

> Essa abordagem pega todas as linhas de cada tabela de streaming, executa uma operação de inner join completa e incorpora otimizações quando aplicável.

Use o código na seção `C. Use a Materialized View to Join Two Streaming Tables` para entender como a visão materializada Gold é criada.

1. Esta instrução cria a visão materializada **3_gold_db.full_order_info_gold_demo5** juntando as seguintes tabelas de streaming:
   - **2_silver_db.status_silver_demo5**
   - **2_silver_db.orders_silver_demo5**

2. As cláusulas `COMMENT` e `TBLPROPERTIES` documentam a visão e a rotulam como Gold na arquitetura medallion.

3. Observe que a palavra-chave `STREAM` não é usada ao referenciar as tabelas de streaming na criação da visão materializada.
   - Isso irá juntar **todos os dados** de ambas as tabelas de streaming e retornar sua visão materializada final.

### E5. Visões Materializadas Gold para `Pedidos Cancelados` e `Pedidos Entregues`

Use o código na seção `D. Criação de Visões Materializadas Gold para Pedidos Cancelados e Entregues` para entender como as visões Gold finais são criadas.

1. Esta seção cria duas visões materializadas Gold a partir de **3_gold_db.full_order_info_gold_demo5**:
   - **3_gold_db.cancelled_orders_gold_demo5**, pedidos cancelados com dias até o cancelamento.
   - **3_gold_db.delivered_orders_gold_demo5**, pedidos entregues com dias até a entrega.

2. Cada visão materializada filtra por um status específico e calcula uma métrica de negócio simples usando `datediff`.
   - Função `datediff`:
[AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/datediff) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/datediff) |
[GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/datediff)

3. As seções `COMMENT` e `TBLPROPERTIES` documentam cada visão e as rotulam como Gold na arquitetura medallion.


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Information
  </strong>
  <div style="color:#333;">

- **Materialized views include built-in optimizations where applicable:**

  - Incremental refresh for materialized views: [AWS](https://docs.databricks.com/aws/en/optimizations/incremental-refresh) |[Azure](https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh) |
  [GCP](https://docs.databricks.com/gcp/en/optimizations/incremental-refresh)

  - [Delta Live Tables Announces New Capabilities and Performance Optimizations](https://www.databricks.com/blog/2022/06/29/delta-live-tables-announces-new-capabilities-and-performance-optimizations.html)

  - [Cost-effective, incremental ETL with serverless compute for Delta Live Tables pipelines](https://www.databricks.com/blog/cost-effective-incremental-etl-serverless-compute-delta-live-tables-pipelines)

- **Stateful joins (Stream to Stream):** For stateful joins in pipelines (i.e., joining incrementally as data is ingested), refer to the Optimize stateful processing in Spark Declarative Pipelines with watermarks documentation:
[AWS](https://docs.databricks.com/aws/en/dlt/stateful-processing) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/stateful-processing) |
[GCP](https://docs.databricks.com/gcp/en/dlt/stateful-processing)
   - **Stateful joins are an advanced topic and outside the scope of this course.**

  </div>
</div>




## F. Create the Production Pipeline
Follow the steps below to modify the pipeline settings and run the production pipeline.

### F1. Revisar e Modificar as Configurações do Pipeline

1. Complete as etapas abaixo no seu **Lakeflow Editor** para configurar seu Spark Declarative Pipeline para **produção**:

   a. Selecione **Configurações** para visualizar as configurações do pipeline.

   b. Na seção **Configurações do Pipeline**, você pode:
      - Modificar o **Nome do Pipeline** e as configurações de **Executar como** (este laboratório não permite modificar o **Executar como**).

         - Se tivesse permissão, poderia selecionar o ícone de lápis ![pencil_settings_icon.png](./Includes/images/common/pencil_settings_icon.png) ao lado de **Executar como** para modificar a opção.

         - Opcionalmente, você pode alterar o executor do pipeline para um principal de serviço. Um principal de serviço é uma identidade criada no Databricks para uso com ferramentas automatizadas, jobs e aplicações.
            - Para mais informações, veja a documentação **O que é um principal de serviço?**: [AWS](https://docs.databricks.com/aws/en/admin/users-groups/service-principals#what-is-a-service-principal) |
            [Azure](https://learn.microsoft.com/en-us/azure/databricks/admin/users-groups/service-principals#what-is-a-service-principal) |
            [GCP](https://docs.databricks.com/gcp/en/admin/users-groups/service-principals#what-is-a-service-principal)

      - Em **Modo do Pipeline**, certifique-se de que **Disparado** está selecionado para que o pipeline execute em uma agenda e processe dados incrementalmente.
        - Alternativamente, você pode escolher o modo **Contínuo** para manter o pipeline rodando o tempo todo.
        - Para mais detalhes, veja **Modo disparado vs. contínuo do pipeline**: [AWS](https://docs.databricks.com/aws/en/dlt/pipeline-mode) |
        [Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/pipeline-mode) |
        [GCP](https://docs.databricks.com/gcp/en/dlt/pipeline-mode)

   c. Na seção **Ativos de código**, confirme que:

      - **Pasta raiz** aponta para este projeto de pipeline (**5 - Deploying a Pipeline to Production Project**).

      - **Código fonte** referencia as pastas **orders** e **status** dentro deste projeto.

   d. Na seção **Local padrão para ativos de dados**, confirme o seguinte:

      - **Catálogo padrão** é o seu catálogo **labuser**.

      - **Schema padrão** é o **default**.

   e. Na seção **Compute**, confirme que o compute **Serverless** está selecionado.

   f. Na seção **Configuração**, certifique-se de que a chave `source` está definida para o caminho do volume da fonte de dados: `/Volumes/labuser_/sdp_1_bronze/source`

2. Na seção **Configurações avançadas** na parte inferior:

   a. Expanda **Configurações avançadas**.

   b. Clique em **Editar configurações avançadas**.

   c. Para **Canal**, você pode deixar como **Atual** para fins de treinamento:
    - **Atual** - Usa a versão estável mais recente do Databricks Runtime, recomendada para produção.
    - **Prévia** - Usa uma versão mais recente, potencialmente menos estável do Runtime, ideal para testar novos recursos.
    - Veja as **notas de versão do Lakeflow Spark Declarative Pipelines** e a documentação do processo de atualização de versão para mais informações: [AWS](https://docs.databricks.com/aws/en/release-notes/dlt/) |
    [Azure](https://learn.microsoft.com/en-us/azure/databricks/release-notes/dlt/) |
    [GCP](https://docs.databricks.com/gcp/en/release-notes/dlt/)


<div style="background: #FFFDE7; border: 2px solid #FFAB00; border-radius: 8px; padding: 16px 20px; font-size: 14pt; line-height: 1.8; color: #0b2026; margin: 8px 0 8px 28px;">
  <div style="font-weight: 700; font-size: 15pt; margin-bottom: 10px;">   d. ⚠️ OBRIGATÓRIO — Na seção <strong>Logs de eventos</strong>:</div>
  <ul style="margin: 0; padding-left: 20px;">
    <li>Selecione <strong>Publicar log de eventos no Unity Catalog</strong>.</li>
    <li><strong>Nome do log de eventos</strong> - <code>event_log_demo_5</code>.</li>
    <li><strong>Catálogo do log de eventos</strong> - catálogo <strong>labuser</strong>.</li>
    <li><strong>Schema do log de eventos</strong> - schema <strong>sdp_1_bronze</strong>.</li>
    <li>Selecione <strong>Salvar</strong>.</li>
  </ul>
  <div style="margin-top: 12px; font-size: 13.5pt; color: #5A6F77;">
    <strong>NOTA:</strong> Se o log de eventos não for salvo no local correto, os próximos passos de exploração do log de eventos não funcionarão corretamente.
  </div>
</div>

3\. Clique em **Salvar** para salvar as configurações do seu pipeline.

### F2. Agendar o Pipeline
1. Quando seu pipeline estiver pronto para produção, você vai querer **agendá-lo para rodar em um intervalo de tempo ou continuamente**.

   Para esta demonstração, vamos:
   - Agendar o pipeline para rodar todos os dias às 20:00.
   - Opcionalmente configurar notificações para alertá-lo ao **Iniciar**, **Sucesso** e **Falha** do job.
     *(Se não quiser notificações por e-mail, pode pular esta etapa.)*

   Complete as etapas abaixo para agendar o pipeline:

   a. Selecione o botão **Agendar** (pode ser um pequeno ícone de calendário se sua tela estiver minimizada).

   b. Para o nome do job, deixe como **5 - Deploying a Pipeline to Production Project - labuser-name**.

   c. Abaixo de **Nome do job**, selecione **Avançado**.

   d. Na seção **Agendamento**, configure o seguinte:
   - Defina o **Dia**.
   - Defina o horário para **20:00** (8:00 PM).
   - Deixe o **Fuso horário** como padrão.
   - Selecione **Mais opções** e, em **Notificações**, adicione seu e-mail para receber alertas de:
     - **Início**
     - **Sucesso**
     - **Falha**

   e. Clique em **Criar** para salvar e agendar o job.

  **NOTA:** Você também pode definir o pipeline para rodar alguns minutos após o horário atual para vê-lo iniciar pelo agendador.

## G. Execute e Visualize o Pipeline Spark Declarative para `orders` e `status`

1. Execute manualmente (em vez de aguardar o job para fins de treinamento) seu Spark Declarative Pipeline e visualize os resultados.
    - **NOTA:** Atualmente temos um arquivo JSON em ambos os volumes **status** e **orders**.

2. Após a conclusão da primeira execução do pipeline, complete o seguinte:

   a. Examine o **gráfico do Pipeline** e confirme:
      - **fluxo status**
         - 536 linhas foram lidas nas tabelas de streaming **status_bronze_demo5** e **status_silver_demo5**
      - **fluxo orders**
         - 174 linhas foram lidas nas tabelas de streaming **orders_bronze_demo5** e **orders_silver_demo5**
         - 7 linhas estão na visão materializada **gold_orders_by_date_demo5**
      - **join de tabelas de streaming e visões materializadas gold**
         - 536 linhas estão na visão materializada **full_order_info_gold_demo5** (JOIN)
            - 8 linhas estão na visão materializada **cancelled_orders_gold_demo5**
            - 94 linhas estão na visão materializada **delivered_orders_gold_demo5**


#### Checkpoint

  ![Final Demo 5 Pipeline](./Includes/images/deploying-a-pipeline-to-production/demo5_pipeline_image_run1.png)

## H. Incrementally Process new Data in your Pipeline

### H1. Land More Files in your Volume
1. Run the cell below to add **4** more JSON files to your volumes to simulate new files being landed into cloud storage:
    - `/Volumes/labuser/sdp_1_bronze/source/orders`
    - `/Volumes/labuser/sdp_1_bronze/source/status`

In [0]:
%python

## Find data in workspace data folder
data_path = find_folder('Includes/data')

## Land JSON files to your orders volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/orders',
    target_volume_path=f'{source_volume_path}/orders',
    n=5
)

## Land JSON files to your status volume
copy_workspace_files_to_volume(
    src_workspace_folder=f'{data_path}/status',
    target_volume_path=f'{source_volume_path}/status',
    n=5
)


  Searching for 'Includes/data' folder...
  Current directory: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines
  Checking: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data... FOUND


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/labuser15140516_1778971530@vocareum.com/build-data-pipelines-with-lakeflow-spark-declarative-pipelines-en_us-3.1.0/Build Data Pipelines with Lakeflow Spark Declarative Pipelines/Includes/data/orders

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders

  STEP 3: Reading source files...
  Found 5 file(s) in source folder.

  STEP 4

In [0]:
%python
orders_list = spark.sql(f"LIST '/Volumes/{my_catalog}/sdp_1_bronze/source/orders'")
status_list = spark.sql(f"LIST '/Volumes/{my_catalog}/sdp_1_bronze/source/status'")
display(orders_list)
display(status_list)

path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders/00.json,00.json,15313,1779232038000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders/01.json,01.json,2201,1779234441000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders/02.json,02.json,2201,1779234443000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders/03.json,03.json,2025,1779234446000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/orders/04.json,04.json,2201,1779234448000


path,name,size,modification_time
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/status/00.json,00.json,40658,1779232044000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/status/01.json,01.json,8468,1779234448000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/status/02.json,02.json,8372,1779234451000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/status/03.json,03.json,7715,1779234453000
/Volumes/labuser15140516_1778971530/sdp_1_bronze/source/status/04.json,04.json,6729,1779234455000


### H2. Run the Pipeline to Incrementally Process New Data
1. After you have landed **4** new files into the data source volumes, **run the pipeline to process the newly landed JSON files**.


2. Após a conclusão do pipeline, visualize a execução do Pipeline. Observe o seguinte:

    - **fluxo status**
        - O fluxo bronze para silver de **status** ingere 410 novas linhas.

    - **fluxo orders**
        - O fluxo bronze para silver de **orders** ingere 98 novas linhas.
        - A visão materializada **orders_by_date_gold_demo5** contém 11 linhas.

    - **join de tabelas de streaming e visões materializadas gold**
        - A visão materializada **full_order_info_gold_demo5** (join) contém um total de 946 linhas (as 536 anteriores + as novas 410 linhas).
        - A visão materializada **cancelled_orders_gold_demo5** contém 21 linhas.
        - A visão materializada **delivered_orders_gold_demo5** contém 176 linhas.

#### Checkpoint - 4 Novos Arquivos
![Pipeline Demo 5](./Includes/images/deploying-a-pipeline-to-production/demo5_pipeline_image_run2.png)

3. Na janela na parte inferior do seu pipeline:

    a. Selecione o link **Expectations** para a tabela **status_silver_demo5**.

    b. Deve conter o valor **1 atendida | 1 não atendida**. Observe que nesta execução, 7,6% (31 linhas) para a expectativa **valid_order_status** retornaram um aviso.

    c. Isso é algo que gostaríamos de investigar e corrigir em futuras etapas do pipeline.

## I. Introdução ao Log de Eventos do Pipeline (Tópico Avançado)

Após executar seu pipeline e publicar com sucesso o log de eventos como uma tabela chamada **event_log_demo_5** no seu schema (banco de dados) **labuser.default**, comece a explorar o log de eventos.

Aqui vamos apresentar rapidamente o log de eventos. **Para processar o log de eventos, você precisará saber como analisar strings formatadas em JSON.**

  - Documentação de Monitoramento de Lakeflow Spark Declarative Pipelines:
[AWS](https://docs.databricks.com/aws/en/dlt/observability) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/observability) |
[GCP](https://docs.databricks.com/gcp/en/dlt/observability)

**RESOLUÇÃO DE PROBLEMAS:**
- **OBRIGATÓRIO:** Se você não executou o pipeline e publicou o log de eventos, o código abaixo não irá funcionar. Certifique-se de completar todas as etapas antes de iniciar esta seção.

- **LOG DE EVENTOS OCULTO:** Por padrão, Spark Declarative Pipelines grava o log de eventos em uma tabela Delta oculta no catálogo e schema padrão configurados para o pipeline. Embora oculta, a tabela ainda pode ser consultada por todos os usuários com privilégios suficientes. Por padrão, apenas o proprietário do pipeline pode consultar a tabela de log de eventos. O nome padrão para o log de eventos oculto é formatado como:

  - `catalog.schema.event_log_{pipeline_id}` - onde o pipeline ID é o UUID atribuído pelo sistema, com traços substituídos por underscores.

  - Consultar o Log de Eventos: [AWS](https://docs.databricks.com/aws/en/ldp/monitor-event-logs#query-the-event-log) |
  [Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/monitor-event-logs#query-event-log) |
  [GCP](https://docs.databricks.com/gcp/en/ldp/monitor-event-logs#query-the-event-log)

1. Complete the following steps to view the **labuser.default.event_log_demo_5** event log in your catalog:

   a. Select the catalog icon ![Catalog Icon](./Includes/images/common/catalog_icon.png) from the left navigation pane.

   b. Expand your **labuser** catalog.

   c. Expand the following schemas (databases):
      - **sdp_1_bronze**
      - **sdp_2_silver**
      - **sdp_3_gold**

   d. Notice the following:
      - In the **sdp_1_bronze**, **sdp_2_silver**, and **sdp_3_gold** schemas, the pipeline streaming tables and materialized views were created (they end with **demo5**).
      - In the **sdp_1_bronze** schema, the pipeline has published the event log as a table named **event_log_demo_5**.

**NOTE:** You might need to refresh the catalogs to view the streaming tables, materialized views, and event log.


2. Query your **labuser.sdp_1_bronze.event_log_demo_5** table to see what the event log looks like.

   Notice that it contains all events within the pipeline as **STRING** columns (typically JSON-formatted strings) or **STRUCT** columns. Databricks supports the `:` (colon) operator to parse JSON fields. See the `:` operator documentation: [AWS](https://docs.databricks.com/aws/en/sql/language-manual/functions/colonsign) | 
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/colonsign) |
   [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/functions/colonsign)

   The following table describes the event log schema. Some fields contain JSON data—such as the **details** field—which must be parsed to perform certain queries.

In [0]:
SELECT *
FROM sdp_1_bronze.event_log_demo_5;

id sequence origin timestamp message level maturity_level error details event_type 2f0b3ee0-53dd-11f1-882b-fe8e80245ff1 List(List(execution, 1779234497435003), 1779234486735001) List(AWS, us-west-2, 7474656847338855, null, cecfc685-f880-4997-90b4-3f0916172f21, WORKSPACE, 5 - Deploying a Pipeline to Production Project - labuser15140516_1778971530, null, eb8901e1-6a99-4028-b809-c041a3619522, null, null, null, null, null, null, eb8901e1-6a99-4028-b809-c041a3619522, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null) 2026-05-19T23:48:06.734Z User labuser15140516_1778971530@vocareum.com started an update. INFO STABLE null {"user_action":{"action":"START","user_name":"labuser15140516_1778971530@vocareum.com","user_id":73868070869994,"request":{"start_request":{"full_refresh":false,"validate_only":false,"explore_only":false,"development":true}}}} user_action 2f0ec150-53dd-11f1-882b-fe8e80245ff1 List(List(execution, 1779234497435004), 1779234486759001) List(AWS, us-west-2, 7474656847338855, null, cecfc685-f880-4997-90b4-3f0916172f21, WORKSPACE, 5 - Deploying a Pipeline to Production Project - labuser15140516_1778971530, null, eb8901e1-6a99-4028-b809-c041a3619522, null, null, null, null, null, null, eb8901e1-6a99-4028-b809-c041a3619522, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null, null) 2026-05-19T23:48:06.745Z Update eb8901 started by USER_ACTION. INFO STABLE null {"create_update":{"cause":"USER_ACTION","config":{"id":"cecfc685-f880-4997-90b4-3f0916172f21","pipeline_type":"WORKSPACE","name":"5 - Deploying a Pipeline to Production Project - labuser15140516_1778971530","configuration":{"pipelines.acfsProcessorVersion":"2","pipelines.allowCatalogManagedPropertyOnMVST":"true","pipelines.allowClearingTableComment":"true","pipelines.allowForwardReferencesInSinkRegistration":"true","pipelines.allowPythonDecoratedDatasetAccessDunderName":"true","pipelines.alterExistingMvSt":"true","pipelines.alterViewsInHMS.enabled":"true","pipelines.alterableMetadataBehavior":"preserve","pipelines.analysis.allowSqlFlowWithConfParallelResolution":"true","pipelines.analysis.enableImmediateFlowFailureEventEmission":"true","pipelines.analysis.maybeResolveFlowsParallely":"true","pipelines.analysis.maybeResolvePureSQLPipelinesParallely":"true","pipelines.analysis.pruneUnresolvedFromJsonDependents":"true","pipelines.applyChanges.Type2IgnoreNullPersistentStateChange":"true","pipelines.applyChanges.Type2OmitNullEntriesInVersionMap.Read":"true","pipelines.applyChanges.Type2OmitNullEntriesInVersionMap.Write":"true","pipelines.applyChanges.applyChangesMultiflowIgnoreNullOffFix.enabled":"true","pipelines.applyChanges.applyChangesMultiflowLatestDeleteVersion.enabled":"true","pipelines.applyChanges.dynamicFilePruning.enabled":"true","pipelines.applyChanges.dynamicPartitionPruning.enabled":"true","pipelines.applyChanges.useAfterImageStatusColumn.enabled":"false","pipelines.applyChangesFromSnapshot.analyzeLambdaSourceWithoutPipelineContext":"true","pipelines.applyChangesFromSnapshot.castSnapshotToTargetSchema":"true","pipelines.applyChangesFromSnapshot.stateStoreFormat":"proto","pipelines.assignSqlStatesToBuiltInPythonErrors":"true","pipelines.asyncServerInit":"true","pipelines.attachEnzymeExpectationsUsingTransformDF":"true","pipelines.attachExpectationWithFunctionResult":"true","pipelines.attachExpectationsUsingTransformDF":"true","pipelines.autoCdc.enableBitemporalAutoCDC":"false","pipelines.autoCdc.logCaseDifferences":"true","pipelines.autoCdcColumnNormalizationRefactor":"true","pipelines.blockDuplicatedUseOfFromJsonSchemaLocationKey":"true","pipelines.brickstore.checkpointFPIPercentageForStreaming":"5","pipelines.brickstore.enableDirectToStorageChecksums":"false","pipelines.brickstore.enableDirectToStorageSseCEncryption":"true","pipelines.brickstore.enableInitialLoadTableSizeCheck":"true","pipelines.brickstore.enableSyncedTableCredentialRefresh":"

| Field          | Description |
|----------------|-------------|
| `id`           | A unique identifier for the event log record. |
| `sequence`     | A JSON document containing metadata to identify and order events. |
| `origin`       | A JSON document containing metadata for the origin of the event, for example, the cloud provider, the cloud provider region, user_id, pipeline_id, or pipeline_type to show where the pipeline was created, either DBSQL or WORKSPACE. |
| `timestamp`    | The time the event was recorded. |
| `message`      | A human-readable message describing the event. |
| `level`        | The event type, for example, INFO, WARN, ERROR, or METRICS. |
| `maturity_level` | The stability of the event schema. The possible values are:<br><br>- **STABLE**: The schema is stable and will not change.<br>- **NULL**: The schema is stable and will not change. The value may be NULL if the record was created before the maturity_level field was added (release 2022.37).<br>- **EVOLVING**: The schema is not stable and may change.<br>- **DEPRECATED**: The schema is deprecated and the pipeline runtime may stop producing this event at any time. |
| `error`        | If an error occurred, details describing the error. |
| `details`      | A JSON document containing structured details of the event. This is the primary field used for analyzing events. |
| `event_type`   | The event type. |

**Event Log Schema:**
[AWS](https://docs.databricks.com/aws/en/ldp/monitor-event-log-schema) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/monitor-event-log-schema) |
[GCP](https://docs.databricks.com/gcp/en/ldp/monitor-event-log-schema)

3. The majority of the detailed information you will want from the event log is located in the **details** column, which is a JSON-formatted string. You will need to parse this column.

   You can find more information in the Databricks documentation on how to query JSON strings: [AWS](https://docs.databricks.com/aws/en/semi-structured/json) |
   [Azure](https://learn.microsoft.com/en-us/azure/databricks/semi-structured/json) |
   [GCP](https://docs.databricks.com/gcp/en/semi-structured/json)

   The code below will:

   - Return the **event_type** column.

   - Return the entire **details** JSON-formatted string.

   - Parse out the **flow_progress** values from the **details** JSON-formatted string, if they exist.

   - Parse out the **user_action** values from the **details** JSON-formatted string, if they exist.


In [0]:
SELECT
  id,
  event_type,
  details,
  details:flow_progress,
  details:user_action
FROM sdp_1_bronze.event_log_demo_5

id event_type details flow_progress user_action 2f0b3ee0-53dd-11f1-882b-fe8e80245ff1 user_action {"user_action":{"action":"START","user_name":"labuser15140516_1778971530@vocareum.com","user_id":73868070869994,"request":{"start_request":{"full_refresh":false,"validate_only":false,"explore_only":false,"development":true}}}} null {"action":"START","user_name":"labuser15140516_1778971530@vocareum.com","user_id":73868070869994,"request":{"start_request":{"full_refresh":false,"validate_only":false,"explore_only":false,"development":true}}} 2f0ec150-53dd-11f1-882b-fe8e80245ff1 create_update {"create_update":{"cause":"USER_ACTION","config":{"id":"cecfc685-f880-4997-90b4-3f0916172f21","pipeline_type":"WORKSPACE","name":"5 - Deploying a Pipeline to Production Project - labuser15140516_1778971530","configuration":{"pipelines.acfsProcessorVersion":"2","pipelines.allowCatalogManagedPropertyOnMVST":"true","pipelines.allowClearingTableComment":"true","pipelines.allowForwardReferencesInSinkRegistration":"true","pipelines.allowPythonDecoratedDatasetAccessDunderName":"true","pipelines.alterExistingMvSt":"true","pipelines.alterViewsInHMS.enabled":"true","pipelines.alterableMetadataBehavior":"preserve","pipelines.analysis.allowSqlFlowWithConfParallelResolution":"true","pipelines.analysis.enableImmediateFlowFailureEventEmission":"true","pipelines.analysis.maybeResolveFlowsParallely":"true","pipelines.analysis.maybeResolvePureSQLPipelinesParallely":"true","pipelines.analysis.pruneUnresolvedFromJsonDependents":"true","pipelines.applyChanges.Type2IgnoreNullPersistentStateChange":"true","pipelines.applyChanges.Type2OmitNullEntriesInVersionMap.Read":"true","pipelines.applyChanges.Type2OmitNullEntriesInVersionMap.Write":"true","pipelines.applyChanges.applyChangesMultiflowIgnoreNullOffFix.enabled":"true","pipelines.applyChanges.applyChangesMultiflowLatestDeleteVersion.enabled":"true","pipelines.applyChanges.dynamicFilePruning.enabled":"true","pipelines.applyChanges.dynamicPartitionPruning.enabled":"true","pipelines.applyChanges.useAfterImageStatusColumn.enabled":"false","pipelines.applyChangesFromSnapshot.analyzeLambdaSourceWithoutPipelineContext":"true","pipelines.applyChangesFromSnapshot.castSnapshotToTargetSchema":"true","pipelines.applyChangesFromSnapshot.stateStoreFormat":"proto","pipelines.assignSqlStatesToBuiltInPythonErrors":"true","pipelines.asyncServerInit":"true","pipelines.attachEnzymeExpectationsUsingTransformDF":"true","pipelines.attachExpectationWithFunctionResult":"true","pipelines.attachExpectationsUsingTransformDF":"true","pipelines.autoCdc.enableBitemporalAutoCDC":"false","pipelines.autoCdc.logCaseDifferences":"true","pipelines.autoCdcColumnNormalizationRefactor":"true","pipelines.blockDuplicatedUseOfFromJsonSchemaLocationKey":"true","pipelines.brickstore.checkpointFPIPercentageForStreaming":"5","pipelines.brickstore.enableDirectToStorageChecksums":"false","pipelines.brickstore.enableDirectToStorageSseCEncryption":"true","pipelines.brickstore.enableInitialLoadTableSizeCheck":"true","pipelines.brickstore.enableSyncedTableCredentialRefresh":"false","pipelines.brickstore.enableSyncedTableTimestampWithTz":"true","pipelines.brickstore.lockTimeoutMillis":"1200000","pipelines.brickstore.pgOffHeapFraction":"0.0","pipelines.bypassDltAnalysisForForeachBatchSql.enabled":"true","pipelines.capturePreExecutionEventLogTable":"true","pipelines.cdc.enableCdcSnapshotProgress":"true","pipelines.cdc.enableGatewayErrorPropagation":"true","pipelines.cdc.skipCdcSnapshotFlowAfterCommitted":"true","pipelines.cdc.stagingDataGCDryRunMode":"false","pipelines.cdc.stagingDataRetentionDays":"25","pipelines.cdcApplier.cdcApplierEnableSCD2WithUserChosenColumns":"true","pipelines.cdcApplier.enableCDCTypeWidening":"false","pipelines.cdcApplier.enableCommunicationVersioning":"true","pipelines.cdcApplier.enableLumberjack":"false","pipelines.cdcApplier.enableNewRealtimeApplier":"false","pipelines.cdcApplier.enableSourceChangeValidation":"true","pipelines.cdcApplier.filterC

4. Um caso de uso do log de eventos é examinar métricas de qualidade de dados para todas as execuções do seu pipeline. Essas métricas fornecem insights valiosos sobre seu pipeline, tanto a curto quanto a longo prazo. As métricas são capturadas para cada restrição durante toda a vida útil da tabela.

   Abaixo está um exemplo de consulta para obter essas métricas. Não vamos nos aprofundar no código de análise de JSON aqui. Este exemplo simplesmente demonstra o que é possível com o **event_log**.

   Execute a célula e observe os resultados. Observe o seguinte:
   - Os **passing_records** para cada restrição são exibidos.
   - Os **failing_records** (WARN) para cada restrição são exibidos.

**NOTA:** Se você selecionou **Executar pipeline com atualização completa da tabela** em algum momento durante seu pipeline, seus resultados incluirão métricas de execuções anteriores, bem como da atualização completa. Lógica adicional é necessária para isolar os resultados após a atualização completa da tabela. Isso está fora do escopo deste curso.

In [0]:
CREATE OR REPLACE TEMPORARY VIEW dq_source_vw AS
SELECT explode(
            from_json(details:flow_progress:data_quality:expectations,
                      "array<struct<name: string, dataset: string, passed_records: int, failed_records: int>>")
          ) AS row_expectations
   FROM sdp_1_bronze.event_log_demo_5
   WHERE event_type = 'flow_progress';


-- View the data
SELECT
  row_expectations.dataset as dataset,
  row_expectations.name as expectation,
  SUM(row_expectations.passed_records) as passing_records,
  SUM(row_expectations.failed_records) as warnings_records
FROM dq_source_vw
GROUP BY row_expectations.dataset, row_expectations.name
ORDER BY dataset;

dataset,expectation,passing_records,warnings_records
labuser15140516_1778971530.sdp_2_silver.orders_silver_demo5,valid_notifications,272,0
labuser15140516_1778971530.sdp_2_silver.orders_silver_demo5,valid_date,272,0
labuser15140516_1778971530.sdp_2_silver.orders_silver_demo5,valid_id,272,0
labuser15140516_1778971530.sdp_2_silver.status_silver_demo5,valid_timestamp,946,0
labuser15140516_1778971530.sdp_2_silver.status_silver_demo5,valid_order_status,885,61


### Summary

This was a quick introduction to the pipeline **event_log**. With the **event_log**, you can investigate all aspects of your pipeline runs to explore the runs as well as create overall reports. Feel free to investigate the **event_log** further on your own.

## Additional Resources

- Lakeflow Spark Declarative Pipelines properties reference:
[AWS](https://docs.databricks.com/aws/en/dlt/properties#dlt-table-properties) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/properties#pipeline-table-properties) |
[GCP](https://docs.databricks.com/gcp/en/dlt/properties#dlt-table-properties)

- Table properties and table options:
[AWS](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-tblproperties) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-tblproperties) |
[GCP](https://docs.databricks.com/gcp/en/sql/language-manual/sql-ref-syntax-ddl-tblproperties)

- Triggered vs. continuous pipeline mode:
[AWS](https://docs.databricks.com/aws/en/dlt/pipeline-mode) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/pipeline-mode) |
[GCP](https://docs.databricks.com/gcp/en/dlt/pipeline-mode)

- Development and production modes:
[AWS](https://docs.databricks.com/aws/en/ldp/updates#development-mode) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/updates#development-mode) |
[GCP](https://docs.databricks.com/gcp/en/ldp/updates#development-mode)

- Monitor Lakeflow Spark Declarative Pipelines:
[AWS](https://docs.databricks.com/aws/en/dlt/observability) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/observability) |
[GCP](https://docs.databricks.com/gcp/en/dlt/observability)

- **Materialized views include built-in optimizations where applicable:**
  - Incremental refresh for materialized views:
[AWS](https://docs.databricks.com/aws/en/optimizations/incremental-refresh) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/optimizations/incremental-refresh) |
[GCP](https://docs.databricks.com/gcp/en/optimizations/incremental-refresh)
  - [Delta Live Tables Announces New Capabilities and Performance Optimizations](https://www.databricks.com/blog/2022/06/29/delta-live-tables-announces-new-capabilities-and-performance-optimizations.html)
  - [Cost-effective, incremental ETL with serverless compute for Delta Live Tables pipelines](https://www.databricks.com/blog/cost-effective-incremental-etl-serverless-compute-delta-live-tables-pipelines)

- **Stateful joins:** For stateful joins in pipelines (i.e., joining incrementally as data is ingested), refer to the Optimize stateful processing in Lakeflow Spark Declarative Pipelines with watermarks documentation:
[AWS](https://docs.databricks.com/aws/en/dlt/stateful-processing) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/ldp/stateful-processing) |
[GCP](https://docs.databricks.com/gcp/en/dlt/stateful-processing). 
  - **Stateful joins are an advanced topic and outside the scope of this course.**


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>